In [ ]:
from pathlib import Path

import prism

from imagematerials import buildings as bld
from imagematerials.concepts import create_building_graph
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericStocks,
    MaterialIntensities,
)
from imagematerials.util import (
    export_to_netcdf,
    import_from_netcdf,
    read_climate_policy_config, 
    read_circular_economy_config
)

circularity_scenario = "base"
climate_scenario = 'SSP2'


In [2]:
def get_buildings_prep():
    climate_policy_scenario_dir = Path("../data/raw") / climate_scenario
    scenario_path = Path("../data/raw") / 'circular_economy_scenarios' / circularity_scenario
    circular_economy_scenario_dirs = {circularity_scenario: scenario_path}
    climate_policy_config = read_climate_policy_config(climate_policy_scenario_dir)
    circular_economy_config = read_circular_economy_config(circular_economy_scenario_dirs)
    base_dir = Path("..", "data", "raw")
    prep_fp = Path("prep_buildings.nc")
    if not prep_fp.is_file():
        prep_data = bld.preprocess(base_dir, climate_policy_config, circular_economy_config)
        export_to_netcdf(prep_data, prep_fp)
    else:
        prep_data = import_from_netcdf(prep_fp)
    new_prep_data = {k: v for k, v in prep_data.items()}
    new_prep_data["knowledge_graph"] = create_building_graph()
    new_prep_data["shares"] = None
    sec_bld = Sector("bld", new_prep_data)
    return sec_bld

In [3]:
if not prep_fp.is_file():
    prep_data = preprocess(base_dir)
    export_to_netcdf(prep_data, prep_fp)
else:
    prep_data = import_from_netcdf(prep_fp)

In [ ]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [ ]:
main_model_factory = ModelFactory(
    new_prep_data, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    ).finish()

In [ ]:
main_model_factory.simulate(simulation_timeline)

list(main_model_factory.bld)